#  TravelMatch: Data-Driven Persona Identification and Package Recommendation
**Business Analytics Individual Case Study**

---

**Author:** Abdus Samad  
**Dataset:** `survey_responses.csv` (Customer Preferences) & `travel_packages_10k.csv` (Inventory)  

### Executive Summary
The travel industry faces a significant challenge in matching the right customer to the right holiday package out of tens of thousands of options. This notebook builds an end-to-end analytical pipeline that:
1. Uses **Unsupervised Learning (K-Means Clustering)** to discover latent customer personas.
2. Uses **Supervised Learning (Random Forest)** to instantly classify new customers into these personas.
3. Implements a **Recommendation Logic** to match users with packages.

Let's dive into the data!

##  1. Data Preparation & Exploration
First, we import our standard data manipulation libraries and `plotly` for interactive visualizations.

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Load the Survey Dataset
survey_df = pd.read_csv('survey_responses.csv')
print(f"Survey Data Shape: {survey_df.shape}")
display(survey_df.head())

Survey Data Shape: (350, 8)


,Respondent_ID,Age_Group,Occupation,Travel_Companions,Budget_Preference,Preferred_Duration_Days,Preferred_Theme,Preferred_Accommodation
0,1,18-25,Student,Family,Mid-range,6,Family tours,Super Deluxe
1,2,60+,Student,Friends,Budget,3,Adventure,Standard
2,3,26-35,Student,Couple,Premium,5,Luxury tours,Luxury
3,4,18-25,Professional,Friends,Mid-range,5,Adventure,Standard
4,5,36-45,Student,Solo,Mid-range,4,Group tours,Deluxe


### Interactive Exploratory Data Analysis (EDA)
Understanding the demographic makeup of our potential customers is crucial.

In [10]:
fig = px.pie(survey_df, names='Budget_Preference', title='Distribution of Customer Budget Preferences',
             color_discrete_sequence=px.colors.sequential.Plasma, hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [11]:
fig = px.histogram(survey_df, x='Preferred_Duration_Days', color='Age_Group', barmode='group',
                   title='Preferred Travel Duration by Age Group',
                   color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_layout(xaxis_title="Days", yaxis_title="Count")
fig.show()

In [12]:
fig = px.sunburst(survey_df, path=['Travel_Companions', 'Preferred_Theme'], 
                  title='Hierarchical View: Companions to Preferred Theme',
                  color='Preferred_Duration_Days', color_continuous_scale='RdBu')
fig.show()

## 2. Discovering Personas via K-Means Clustering
Instead of guessing, we let the data tell us how many customer archetypes exist. We encode our categorical variables and use the **Elbow Method** to find the optimal number of clusters (`K`).

In [13]:
# Convert categorical to numeric for clustering
le = LabelEncoder()
survey_df['Age_Code'] = le.fit_transform(survey_df['Age_Group'])
budget_mapping = {'Budget': 1, 'Mid-range': 2, 'Premium': 3}
survey_df['Budget_Code'] = survey_df['Budget_Preference'].map(budget_mapping)

features_for_clustering = ['Age_Code', 'Budget_Code', 'Preferred_Duration_Days']
X_cluster = survey_df[features_for_clustering]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow Method Calculation
inertia = []
K_range = range(1, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

fig = px.line(x=list(K_range), y=inertia, markers=True, 
              title='Elbow Method for Optimal K', 
              labels={'x':'Number of Clusters (K)', 'y':'Inertia'})
fig.add_vline(x=4, line_dash="dash", line_color="red", annotation_text="Optimal K=4")
fig.show()

The Elbow plot flattens out around K=4, suggesting 4 distinct customer personas. Let's fit the model and use Principal Component Analysis (PCA) to visualize these multi-dimensional clusters in a 3D interactive scatter plot.

In [14]:
# Fit K-Means with K=4
kmeans = KMeans(n_clusters=4, random_state=42)
survey_df['Persona_Cluster'] = kmeans.fit_predict(X_scaled)

# PCA for 3D Visualization
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
survey_df['PCA1'] = X_pca[:, 0]
survey_df['PCA2'] = X_pca[:, 1]
survey_df['PCA3'] = X_pca[:, 2]

survey_df['Persona_Cluster_Name'] = survey_df['Persona_Cluster'].map({
    0: 'Persona 0: Budget Backpackers',
    1: 'Persona 1: Luxury Couples',
    2: 'Persona 2: Family Vacationers',
    3: 'Persona 3: Premium Explorers'
})

fig = px.scatter_3d(survey_df, x='PCA1', y='PCA2', z='PCA3', 
                    color='Persona_Cluster_Name', symbol='Budget_Preference',
                    title='3D Interactive Cluster Map (PCA Reduced)',
                    opacity=0.7, size_max=10)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig.show()

##  3. Predicting Personas with Random Forest
While K-Means is great for discovery, running it on every new user is inefficient. Instead, we'll train a **Random Forest Classifier** to learn the rules of these clusters and instantly classify new survey takers.

In [15]:
# Prepare data (One-Hot Encoding for categorical variables)
df_encoded = pd.get_dummies(survey_df, columns=['Travel_Companions', 'Preferred_Theme', 'Age_Group', 'Budget_Preference'], drop_first=True)
cols_to_drop = ['Respondent_ID', 'Occupation', 'Preferred_Accommodation', 
                'Persona_Cluster', 'Persona_Cluster_Name', 'PCA1', 'PCA2', 'PCA3', 'Age_Code', 'Budget_Code']
X_class = df_encoded.drop([c for c in cols_to_drop if c in df_encoded.columns], axis=1)
y_class = df_encoded['Persona_Cluster']

X_train, X_test, y_train, y_test = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

y_pred = rf_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"⭐ Random Forest Classifier Accuracy: {acc*100:.2f}%\n")
print(classification_report(y_test, y_pred))

⭐ Random Forest Classifier Accuracy: 94.29%

              precision    recall  f1-score   support

           0       0.93      0.97      0.95        29
           1       1.00      0.95      0.97        19
           2       0.88      0.78      0.82         9
           3       0.93      1.00      0.96        13

    accuracy                           0.94        70
   macro avg       0.93      0.92      0.93        70
weighted avg       0.94      0.94      0.94        70



###  Interpreting the Model (Feature Importance)
Let's visualize which factors the Random Forest considers most important when classifying a customer.

In [16]:
feature_importances = pd.DataFrame({'Feature': X_class.columns, 'Importance': rf_clf.feature_importances_})
feature_importances = feature_importances.sort_values('Importance', ascending=True).tail(10)

fig = px.bar(feature_importances, x='Importance', y='Feature', orientation='h',
             title='Top 10 Most Deterministic Features for Persona Classification',
             color='Importance', color_continuous_scale='Viridis')
fig.show()

##  4. Business Recommendations & Conclusion
Based on our models, the `Preferred_Duration_Days` and the `Budget` tiers absolutely dominate the decision-making process.

**Strategic Actions:**
1. **Dynamic Website Generation:** Implement the Random Forest model on the front-end. When a user inputs their Budget, Duration, and Companions, dynamically render only the vacation packages tailored to their predicted Persona.
2. **Marketing Optimization:** Reduce ad-spend on generic "all-inclusive" campaigns. Instead, launch highly targeted campaigns (e.g., "Quick Weekend Getaways" directed towards Persona 0, and "Luxury Month-Long Honeymoons" for Persona 1).